# Tensor NPZ Inspection Notebook

This notebook lets you select one `.npz` tensor export using `ipywidgets`, inspect raw array values, and render plots that match the preprocessing workflow style (heatmaps for matrices, per-window slice views for 3D tensors, and occupancy scatter views where relevant).

Run cells top-to-bottom.


In [1]:
# Cell 2 - Imports and global setup
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

try:
    import pandas as pd
except Exception:
    pd = None

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True


def detect_repo_root(start: Path) -> Path:
    """Walk up from the current directory until a folder containing `preprocessing` is found."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "preprocessing").exists():
            return candidate
    return start


REPO_ROOT = detect_repo_root(Path.cwd())
DEFAULT_NPZ_SEARCH_ROOTS = [
    REPO_ROOT / "preprocessing" / "03.training-export" / "output",
    REPO_ROOT / "preprocessing",
]

CURRENT_NPZ_PATH = None
CURRENT_NPZ_DATA = {}

print(f"Repo root: {REPO_ROOT}")


Repo root: /home/mitch/development/raccoon-ball


In [2]:
# Cell 3 - NPZ discovery, loading, and summary table

def find_npz_files(search_roots, limit=5000):
    """Return sorted list of .npz paths discovered under one or more roots."""
    paths = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("*.npz"):
            if path.is_file():
                paths.append(path.resolve())
                if len(paths) >= limit:
                    break
        if len(paths) >= limit:
            break
    unique_sorted = sorted({p for p in paths})
    return unique_sorted


def safe_min_max(arr: np.ndarray):
    """Return (min, max) when numerical/boolean; otherwise (None, None)."""
    if arr.size == 0:
        return None, None

    kind = arr.dtype.kind
    if kind in {"U", "S", "O"}:
        return None, None

    try:
        return float(np.nanmin(arr)), float(np.nanmax(arr))
    except Exception:
        return None, None


def summarize_npz_dict(data: dict):
    """Display high-level summary of all arrays in the selected NPZ file."""
    rows = []
    for key, arr in data.items():
        mn, mx = safe_min_max(arr)
        rows.append(
            {
                "key": key,
                "shape": tuple(arr.shape),
                "dtype": str(arr.dtype),
                "ndim": int(arr.ndim),
                "size": int(arr.size),
                "min": mn,
                "max": mx,
            }
        )

    if not rows:
        display(Markdown("No arrays found in this NPZ file."))
        return

    if pd is not None:
        df = pd.DataFrame(rows).sort_values(["ndim", "key"]).reset_index(drop=True)
        display(df)
    else:
        for row in rows:
            print(row)


def load_npz_file(npz_path: Path):
    """Load arrays from NPZ into in-memory dict and update globals."""
    global CURRENT_NPZ_PATH, CURRENT_NPZ_DATA

    npz_path = Path(npz_path).resolve()
    with np.load(npz_path, allow_pickle=False) as payload:
        data = {key: payload[key] for key in payload.files}

    CURRENT_NPZ_PATH = npz_path
    CURRENT_NPZ_DATA = data

    display(Markdown(f"### Loaded `{npz_path.relative_to(REPO_ROOT)}`"))
    summarize_npz_dict(CURRENT_NPZ_DATA)

    if "refresh_tensor_key_options" in globals():
        refresh_tensor_key_options()


In [3]:
# Cell 4 - Widget UI to select and load a .npz file
npz_path_filter = widgets.Text(
    value="",
    description="Path filter:",
    placeholder="optional substring, e.g. -6db-pump/id_00",
    layout=widgets.Layout(width="95%"),
)

npz_select = widgets.Select(
    options=[],
    value=None,
    rows=14,
    description=".npz files",
    layout=widgets.Layout(width="95%"),
)

refresh_button = widgets.Button(description="Refresh file list", button_style="")
load_button = widgets.Button(description="Load selected .npz", button_style="primary")
status_output = widgets.Output()


def refresh_npz_options(*_):
    paths = find_npz_files(DEFAULT_NPZ_SEARCH_ROOTS)
    filter_text = npz_path_filter.value.strip().lower()

    if filter_text:
        paths = [p for p in paths if filter_text in str(p).lower()]

    options = [(str(p.relative_to(REPO_ROOT)), str(p)) for p in paths]
    npz_select.options = options
    npz_select.value = options[0][1] if options else None

    with status_output:
        clear_output(wait=True)
        print(f"Discovered {len(options)} matching .npz files.")


def on_load_clicked(*_):
    with status_output:
        clear_output(wait=True)
        if not npz_select.value:
            print("No .npz file selected.")
            return

        selected = Path(npz_select.value)
        try:
            load_npz_file(selected)
        except Exception as exc:
            print(f"Failed to load {selected}: {exc}")


refresh_button.on_click(refresh_npz_options)
load_button.on_click(on_load_clicked)

controls_row = widgets.HBox([refresh_button, load_button])

display(Markdown("### Select one tensor export (`.npz`)"))
display(npz_path_filter)
display(controls_row)
display(npz_select)
display(status_output)

refresh_npz_options()


### Select one tensor export (`.npz`)

Text(value='', description='Path filter:', layout=Layout(width='95%'), placeholder='optional substring, e.g. -…

Select(description='.npz files', layout=Layout(width='95%'), options=(), rows=14, value=None)

Output()

In [4]:
# Cell 5 - Raw tensor inspection and plotting utilities

def format_raw_array(arr: np.ndarray, force_full=False, threshold=2000, edgeitems=4):
    """Convert array to a readable string; optionally include full array content."""
    if force_full:
        return np.array2string(arr, threshold=arr.size, edgeitems=edgeitems, max_line_width=180)
    return np.array2string(arr, threshold=threshold, edgeitems=edgeitems, max_line_width=180)


def plot_1d(arr: np.ndarray, key: str):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(arr)
    ax.set_title(f"{key} (1D line plot)")
    ax.set_xlabel("index")
    ax.set_ylabel("value")
    plt.show()


def plot_2d(arr: np.ndarray, key: str, cmap="magma"):
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(arr, aspect="auto", origin="lower", cmap=cmap)
    ax.set_title(f"{key} (2D heatmap)")
    ax.set_xlabel("x / frame")
    ax.set_ylabel("y / band")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    plt.show()


def plot_3d_tensor(arr: np.ndarray, key: str, index0: int, cmap="magma"):
    idx = int(np.clip(index0, 0, arr.shape[0] - 1))
    slice_2d = arr[idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    im0 = axes[0].imshow(slice_2d, aspect="auto", origin="lower", cmap=cmap)
    axes[0].set_title(f"{key}[{idx}, :, :] (slice heatmap)")
    axes[0].set_xlabel("frame")
    axes[0].set_ylabel("band")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)

    max_projection = np.max(arr.astype(np.float32), axis=0)
    im1 = axes[1].imshow(max_projection, aspect="auto", origin="lower", cmap=cmap)
    axes[1].set_title(f"max({key}, axis=0) (projection heatmap)")
    axes[1].set_xlabel("frame")
    axes[1].set_ylabel("band")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)

    plt.tight_layout()
    plt.show()

    if arr.dtype == np.bool_ or np.issubdtype(arr.dtype, np.integer):
        active = np.argwhere(arr > 0)
        if active.size > 0:
            max_points = 30000
            if active.shape[0] > max_points:
                rng = np.random.default_rng(7)
                active = active[rng.choice(active.shape[0], size=max_points, replace=False)]

            fig = plt.figure(figsize=(8, 6))
            ax = fig.add_subplot(111, projection="3d")
            ax.scatter(active[:, 2], active[:, 1], active[:, 0], s=2, alpha=0.25)
            ax.set_title(f"{key} occupancy scatter (sampled active voxels)")
            ax.set_xlabel("frame")
            ax.set_ylabel("band")
            ax.set_zlabel("window")
            plt.show()


def plot_generic(arr: np.ndarray, key: str, index0: int):
    if arr.ndim == 0:
        display(Markdown(f"Scalar value: `{arr.item()}`"))
    elif arr.ndim == 1:
        plot_1d(arr, key)
    elif arr.ndim == 2:
        plot_2d(arr, key)
    elif arr.ndim == 3:
        plot_3d_tensor(arr, key, index0=index0)
    else:
        flat = arr.ravel().astype(np.float32) if arr.dtype.kind in {"b", "i", "u", "f"} else None
        display(Markdown(f"`{key}` has ndim={arr.ndim}; showing histogram of flattened values."))
        if flat is not None and flat.size > 0:
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.hist(flat, bins=80)
            ax.set_title(f"{key} flattened value histogram")
            ax.set_xlabel("value")
            ax.set_ylabel("count")
            plt.show()


In [5]:
# Cell 6 - Interactive tensor browser (raw + plot)
tensor_key_dropdown = widgets.Dropdown(options=[], description="Tensor key:", layout=widgets.Layout(width="70%"))
full_raw_checkbox = widgets.Checkbox(value=False, description="Show full raw array (can be large)")
raw_threshold_slider = widgets.IntSlider(value=1200, min=100, max=20000, step=100, description="Raw threshold:", continuous_update=False)
index0_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Slice idx:", continuous_update=False)
render_button = widgets.Button(description="Render tensor", button_style="primary")
inspect_output = widgets.Output()


def refresh_tensor_key_options():
    keys = sorted(CURRENT_NPZ_DATA.keys()) if CURRENT_NPZ_DATA else []
    tensor_key_dropdown.options = keys
    tensor_key_dropdown.value = keys[0] if keys else None
    update_slice_slider()


def update_slice_slider(*_):
    key = tensor_key_dropdown.value
    if not key or key not in CURRENT_NPZ_DATA:
        index0_slider.disabled = True
        index0_slider.max = 0
        index0_slider.value = 0
        return

    arr = CURRENT_NPZ_DATA[key]
    if arr.ndim == 3:
        index0_slider.disabled = False
        index0_slider.max = max(0, arr.shape[0] - 1)
        index0_slider.value = min(index0_slider.value, index0_slider.max)
    else:
        index0_slider.disabled = True
        index0_slider.max = 0
        index0_slider.value = 0


def on_render_clicked(*_):
    with inspect_output:
        clear_output(wait=True)

        if not CURRENT_NPZ_DATA:
            print("Load an .npz file first.")
            return

        key = tensor_key_dropdown.value
        if key is None:
            print("No tensor key selected.")
            return

        arr = CURRENT_NPZ_DATA[key]

        display(Markdown(f"### `{key}`"))
        display(Markdown(f"shape={arr.shape}, dtype={arr.dtype}, ndim={arr.ndim}, size={arr.size}"))

        raw_text = format_raw_array(
            arr,
            force_full=full_raw_checkbox.value,
            threshold=raw_threshold_slider.value,
        )
        print("Raw contents:")
        print(raw_text)

        if arr.ndim <= 3:
            plot_generic(arr, key, index0=index0_slider.value)
        else:
            plot_generic(arr, key, index0=0)


tensor_key_dropdown.observe(update_slice_slider, names="value")
render_button.on_click(on_render_clicked)

ui = widgets.VBox([
    tensor_key_dropdown,
    widgets.HBox([full_raw_checkbox, raw_threshold_slider]),
    index0_slider,
    render_button,
])

display(Markdown("### Browse arrays inside loaded NPZ"))
display(ui)
display(inspect_output)

refresh_tensor_key_options()


### Browse arrays inside loaded NPZ

Output()

In [6]:
# Cell 7 - Height-bin, mask, and normalized-window consistency checks
from IPython.display import Markdown, display
import numpy as np

if not CURRENT_NPZ_DATA:
    display(Markdown("Load an `.npz` file first using the selector above."))
else:
    required = ("height_bins", "active_mask", "normalized_window")
    missing = [k for k in required if k not in CURRENT_NPZ_DATA]
    if missing:
        display(Markdown(f"Missing expected key(s): {missing}"))
    else:
        height_bins = CURRENT_NPZ_DATA["height_bins"]
        active_mask = CURRENT_NPZ_DATA["active_mask"]
        normalized_window = CURRENT_NPZ_DATA["normalized_window"]

        if height_bins.ndim < 3 or active_mask.ndim < 3 or normalized_window.ndim < 3:
            display(Markdown(
                "Unexpected dimensionality: "
                f"height_bins.ndim={height_bins.ndim}, "
                f"active_mask.ndim={active_mask.ndim}, "
                f"normalized_window.ndim={normalized_window.ndim}."
            ))
        else:
            hb0 = height_bins[0]
            am0 = active_mask[0]
            nw0 = normalized_window[0]

            print("height_bins[0].shape:", hb0.shape)
            print("active_mask[0].shape:", am0.shape)
            print("normalized_window[0].shape:", nw0.shape)
            print("active_mask[0].sum():", int(am0.sum()))
            print("height_bins[0].min() and height_bins[0].max():", int(hb0.min()), int(hb0.max()))
            print("normalized_window[0].min() and normalized_window[0].max():", float(np.min(nw0)), float(np.max(nw0)))

            # Alignment checks: zeros in height bins vs inactive mask
            zeros_hb0 = (hb0 == 0)
            inactive_am0 = (~am0)

            aligned_zeros_with_inactive = np.array_equal(zeros_hb0, inactive_am0)
            zeros_that_are_active = int(np.logical_and(zeros_hb0, am0).sum())
            nonzero_that_are_inactive = int(np.logical_and(hb0 != 0, inactive_am0).sum())

            total_cells = hb0.size
            true_count = int(am0.sum())
            false_count = int(total_cells - true_count)

            print("\nMask density:")
            print(f"True count:  {true_count} / {total_cells} ({100.0 * true_count / total_cells:.2f}%)")
            print(f"False count: {false_count} / {total_cells} ({100.0 * false_count / total_cells:.2f}%)")

            print("\nDo zeros in height_bins[0] line up with active_mask[0] == False?")
            print("Exact alignment:", aligned_zeros_with_inactive)
            print("Zero bins where mask is True:", zeros_that_are_active)
            print("Non-zero bins where mask is False:", nonzero_that_are_inactive)

            # normalized_window agreement checks
            same_shape = (nw0.shape == hb0.shape == am0.shape)
            nw_zero = np.isclose(nw0, 0.0)
            nw_positive = nw0 > 0

            zero_nw_when_inactive = bool(np.all(nw_zero[inactive_am0]))
            no_positive_nw_when_inactive = bool(np.all(~nw_positive[inactive_am0]))
            positive_nw_where_active = int(np.logical_and(nw_positive, am0).sum())
            zero_nw_where_active = int(np.logical_and(nw_zero, am0).sum())

            print("\nDoes normalized_window[0] agree with height_bins[0] and active_mask[0]?")
            print("Shape agreement (nw0 == hb0 == am0):", same_shape)
            print("All inactive-mask cells have normalized_window == 0:", zero_nw_when_inactive)
            print("No inactive-mask cells have normalized_window > 0:", no_positive_nw_when_inactive)
            print("Active cells with normalized_window > 0:", positive_nw_where_active)
            print("Active cells with normalized_window == 0:", zero_nw_where_active)

            # Height bins and normalized window should both be zero where inactive
            hb_zero_when_inactive = bool(np.all((hb0 == 0)[inactive_am0]))
            print("All inactive-mask cells have height_bins == 0:", hb_zero_when_inactive)


Load an `.npz` file first using the selector above.